# Saving ASSIST simulation files

Rebound has the ability to save `SimulationArchive`'s. However, because of the way ASSIST's `integrate_or_interpolate` works (and specifically how it uses data *across* REBOUND steps.

We will validate this by:
1. doing a traditional ASSIST simulation, `integrate_or_interpolate`ing for a set number of times, and
2. doing a manual step-through simulation, saving it to a file, loading it back up, and calling `integrate_or_interpolate` on the same timesteps,
3. using the C function `assist_create_interpolated_simulation` to use the correct accelerations between steps (see: https://github.com/matthewholman/assist/commit/d94d933357a7dee78347d5e1530d5a641c621bac)

Then, we'll compare the data.

First, let's import REBOUND, ASSIST as well as numpy and matplotlib.

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import rebound
import assist

ephem = assist.Ephem("../data/linux_p1550p2650.440", "../data/sb441-n16.bsp")

We create a simple rebound simulation and attach ASSIST.

In [28]:
T0 = 2458849.5
T0_sim = T0 - ephem.jd_ref

def setup_sim():
    sim = rebound.Simulation()
    ax = assist.Extras(sim, ephem)
    sim.add("2024 YR4", date=f"JD{T0}")
    sim.t = T0_sim
    return sim, ax

sim, ax = setup_sim()

print(f"Initialized simulation at t={sim.t}, initial position: {sim.particles[0].xyz}")

Searching NASA Horizons for '2024 YR4'... 
Found: (2024 YR4) 
Initialized simulation at t=7304.5, initial position: [-0.5145660162497446, -2.8176650229161004, -1.26053378304063]


Our endpoint will be N days in the future, selected by SIM_LEN.

In [29]:
SIM_LEN = 3000
T_END = T0_sim + SIM_LEN

times = np.linspace(T0_sim, T_END, SIM_LEN, endpoint=False)
print(f"We'll work with the times {times[0:3]}...{times[-3:]}")
positions_default = np.zeros((SIM_LEN, 3))

sim1, ax1 = setup_sim()
for i, t in enumerate(times):
    ax1.integrate_or_interpolate(t)
    positions_default[i] = sim1.particles[0].xyz

print(f"Finished standard method simulation...sample:\n{positions_default[0:5]}...")

We'll work with the times [7304.5 7305.5 7306.5]...[10301.5 10302.5 10303.5]
Searching NASA Horizons for '2024 YR4'... 
Found: (2024 YR4) 
Finished standard method simulation...sample:
[[-0.51456602 -2.81766502 -1.26053378]
 [-0.50724626 -2.81389356 -1.2584097 ]
 [-0.49992162 -2.81009494 -1.25627347]
 [-0.49259214 -2.80626905 -1.25412505]
 [-0.48525789 -2.80241579 -1.2519644 ]]...


In [30]:
sim2, ax2 = setup_sim()
sim2.save_to_file("sim2.bin", step=1, delete_file=True)
ax2.integrate_or_interpolate(T_END+1)

Searching NASA Horizons for '2024 YR4'... 
Found: (2024 YR4) 


Now test the file...

In [31]:
sim_f = rebound.Simulation("sim2.bin")
ax_f = assist.Extras(sim_f, ephem)

positions_fromfile = np.zeros((SIM_LEN, 3))

for i, t in enumerate(times):
    ax_f.integrate_or_interpolate(t)
    positions_fromfile[i] = sim_f.particles[0].xyz

print(f"Finished interpolating from SimulationArchive...sample:\n{positions_fromfile[0:5]}...\n")
print(f"Average difference between the states: {np.average(positions_default - positions_fromfile)}")

Finished interpolating from SimulationArchive...sample:
[[-0.51456602 -2.81766502 -1.26053378]
 [-0.50724626 -2.81389356 -1.2584097 ]
 [-0.49992162 -2.81009494 -1.25627347]
 [-0.49259214 -2.80626905 -1.25412505]
 [-0.48525789 -2.80241579 -1.2519644 ]]...

Average difference between the states: -3.77063658508342e-13


In [27]:
from rebound.simulationarchive import Simulationarchive
sa=Simulationarchive("sim2.bin")

positions_fromfile_fixed = np.zeros((SIM_LEN-1,3))
for i, t in enumerate(times[1:]):
    tempsim = assist.tools.assist_create_interpolated_simulation(sa, t)
    positions_fromfile_fixed[i] = tempsim.particles[0].xyz

print(f"Average difference between the states: {np.average(positions_default[1:] - positions_fromfile_fixed)}")

Average difference between the states: -8.970580559789846e-17


Note the difference from the C function is better, while the naive loading of the archive is wrong (this is for the extreme case of 2024 YR4, but nonetheless true).

Also note that because we need the acceleration from a step prior, we cannot interpolate during the first timestep (this is hard-coded in the C function, though I'm not sure if it's because it's impossible or inconvenient/incompatible with out SimulationArchives are saved currently).